# GENERATE NEW NAMES USING AI. PARTICULARLY MLP
***

Preparing data

In [228]:
with open("data/names.txt","r") as file:
    data = file.read().splitlines()

In [229]:
alphabets = list(set("".join(data)))
alphabets.sort()

stoi = {k:v for v,k in enumerate(alphabets)}
stoi["."] = 26

itos = {v:k for k,v in stoi.items()}

In [230]:
#Iterate through words

for word in data[:1]:
    chars = "..."+ word + "."

    for char1,char2,char3,char4 in zip(chars,chars[1:],chars[2:],chars[3:]):
        print(char1,char2,char3,"->",char4)

. . . -> e
. . e -> m
. e m -> m
e m m -> a
m m a -> .


# Creating MLP dataset

In [231]:
import torch
import torch.nn.functional as F
import random

In [232]:
Xs = []
ys = []

random.shuffle(data)
for word in data:
    chars = "..."+ word + "."

    for char1,char2,char3,char4 in zip(chars,chars[1:],chars[2:],chars[3:]):
        idx1 = stoi[char1]
        idx2 = stoi[char2]
        idx3 = stoi[char3]
        idx4 = stoi[char4]

        Xs.append([idx1,idx2,idx3])
        ys.append(idx4)



Xs = torch.tensor(Xs)
ys = torch.tensor(ys)

vocab_size = len(stoi)

In [233]:
X_enc = F.one_hot(Xs,num_classes=vocab_size).to(torch.float)
Y_enc = F.one_hot(ys,num_classes=vocab_size).to(torch.float)



In [234]:
tr_idx = int(0.8 * Xs.shape[0])
val_idx = int(0.9 * Xs.shape[0])

X_tr,y_tr = X_enc[:tr_idx], Y_enc[:tr_idx]
X_val, y_val = X_enc[tr_idx:val_idx], Y_enc[tr_idx:val_idx]
X_ts, y_ts = X_enc[val_idx:], Y_enc[val_idx:]

In [235]:
print(f"{X_val.shape= }")
print(f"{X_tr.shape= }")

X_val.shape= torch.Size([22815, 3, 27])
X_tr.shape= torch.Size([182516, 3, 27])


In [236]:
feature_space = 10
batch_size = 10
h_in = feature_space * 3
h_out = 300

Emb = torch.randn((vocab_size,feature_space))
W1 = torch.randn((h_in,h_out)) * 0.2
b1 = torch.randn((1,h_out)) * 0.1

W2 = torch.randn((h_out,vocab_size)) * 0.01
b2 = torch.randn((1,vocab_size)) * 0


parameters = [Emb,W1,b1,W2,b2]

for p in parameters:
    p.requires_grad = True

batch_idx = torch.randint(0,Xs.shape[0],(batch_size,))

X_enc[batch_idx].shape

torch.Size([10, 3, 27])

In [237]:
print(f"Size of model : {sum(p.numel() for p in parameters)}")

Size of model : 17697


In [238]:
# forward pass

batch_idx = torch.randint(0,Xs.shape[0],(batch_size,))
X = X_enc[batch_idx]
y = Y_enc[batch_idx]
X_emb = X @ Emb
h_preact = X_emb.view((-1,h_in)) @ W1  + b1
h = torch.tanh(h_preact)

o = h @ W2 + b2
loss = F.cross_entropy(o,y)

In [239]:
loss.item()

3.3026397228240967

In [240]:
#Full Training

epochs = 200000
log_every = 5000
alpha = [0.1,0.01]
batch_size = 512

for epoch in range(epochs):
    batch_idx = torch.randint(0,X_tr.shape[0],(batch_size,))
    X = X_tr[batch_idx]
    y = y_tr[batch_idx]
    X_emb = X @ Emb
    h_preact = X_emb.view((-1,h_in)) @ W1  + b1
    h = torch.tanh(h_preact)

    o = h @ W2 + b2
    loss = F.cross_entropy(o,y)

    # Zero grad

    for p in parameters:
        p.grad = None

    loss.backward()

    # Update Weights

    for p in parameters:
        p.data -= alpha[epoch//100000] * p.grad

    if epoch % log_every == 0:
        print(f"Epoch : {epoch} Loss : {loss.item()} ")

    


Epoch : 0 Loss : 3.301632881164551 


Epoch : 5000 Loss : 2.1517412662506104 
Epoch : 10000 Loss : 2.154893636703491 
Epoch : 15000 Loss : 2.020233631134033 
Epoch : 20000 Loss : 2.0732455253601074 
Epoch : 25000 Loss : 2.037493944168091 
Epoch : 30000 Loss : 2.188234329223633 
Epoch : 35000 Loss : 2.096416711807251 
Epoch : 40000 Loss : 1.9998246431350708 
Epoch : 45000 Loss : 1.874859094619751 
Epoch : 50000 Loss : 1.980796456336975 
Epoch : 55000 Loss : 2.039558172225952 
Epoch : 60000 Loss : 1.9658888578414917 
Epoch : 65000 Loss : 2.0341832637786865 
Epoch : 70000 Loss : 2.171220064163208 
Epoch : 75000 Loss : 1.9697679281234741 
Epoch : 80000 Loss : 1.9716240167617798 
Epoch : 85000 Loss : 2.020732879638672 
Epoch : 90000 Loss : 2.024237871170044 
Epoch : 95000 Loss : 1.9768027067184448 
Epoch : 100000 Loss : 2.0632550716400146 
Epoch : 105000 Loss : 1.9717490673065186 
Epoch : 110000 Loss : 2.1285948753356934 
Epoch : 115000 Loss : 1.8962082862854004 
Epoch : 120000 Loss : 1.9640809297561646 
Epoch : 125000 Loss : 1

In [241]:
# Evaluate on val set


X_val_emb = X_val @ Emb
h_preact = X_val_emb.view((-1,h_in)) @ W1 + b1
h = torch.tanh(h_preact)
o = h @ W2 + b2
loss = F.cross_entropy(o,y_val)

print(f"Evaluation on val set : Loss : {loss.item()}")

Evaluation on val set : Loss : 2.101027727127075


In [ ]:
class Linear(nn.Module):
    def __init__(self,fan_in,fan_out,bias = True):
        self.W = torch.randn(fan_in,fan_out)
        if bias:
            self.b = torch.randn(1,fan_out)
        else:
            self.b = None

    def forward(self,x):
        if self.b:
            return  x @ self.W + self.b
        else:
            return  x @ self.W